In [1]:
# =========================================================================
# EO STEADY-STATE DIGITAL-TWIN + MINLP OPTIMIZER — configuration cell
# =========================================================================
# Pipeline: ingest PI snapshot -> build inferred tags -> build GEKKO model
#           -> 3-stage solve (IPOPT reconcile -> IPOPT scenario -> APOPT MINLP)
#           -> export actual vs. optimized vs. inferred comparison.
#
# PYTHON-SIDE OVERRIDES APPLIED (see tagged blocks in cells 4 and 6):
#   [FIX-1]  Integer-flag sanitation — cell 4
#            Excel `variables.flag_integer` column is inconsistent: many
#            continuous flow/power vars (e.g. BFW_C_Turb_Steam, BLR_1_HPS_Gen,
#            FD_Fan_BLR_1_Steam) are mistakenly flagged integer, and several
#            status bits are not. We force integer=True for *_Status and
#            counter vars, and integer=False for everything else.
#            EXCEL FIX: set flag_integer correctly in variables sheet.
#
#   [FIX-2]  Physics repair — cell 6 (status <-> flow link)
#            Excel `constraints` rows 46 and 47 are multiplied by *0 and are
#            therefore dormant; without them, DMW_Makeup can stay at 220 t/hr
#            while all DMW pump statuses are 0. We inject:
#              DMW_Makeup <= (Turb_A + Motor_B + Turb_C) * DMW_Pump_Rated_Flow
#              sum(BFW drive statuses) >= 1
#            EXCEL FIX: rewrite row 46 as
#              [DMW_Makeup] <= ([DMW_Turbine_A_Status]+[DMW_Pump_Motor_B_Status]
#                               +[DMW_Turbine_C_Status]) * [DMW_Pump_Rated_Flow]
#            and delete the *0 block in row 47 (or replace with the BFW rule).
#
#   [FIX-3]  Imbalance tags routed to m.Param (frozen at PI) — cell 4
#            These are sensor-drift correction terms. Treat them as constants
#            from PI snapshot rather than decision variables, except
#            BFW_Imbalance which is intentionally left as a variable bounded
#            by +/-20 t/hr (see Task 5b).
#            EXCEL FIX: none needed — this is a pipeline design choice.
# =========================================================================

# -- PATHS AND DATA SOURCES ------------------------------------------------
FEATURE_FILE      = "feature_file_eo_v7_unified.xlsx"   # the configuration/PI workbook
OUTPUT_DIR        = "./output"
PI_ROW_SELECTION  = "best"   # "best" | "first" | "last" | int(row index)

# -- IMBALANCE TAGS: PINNED TO PI SNAPSHOT (FROZEN AS m.Param) -------------
# Rationale: header-level mass/energy sensor drift. Freezing them keeps the
# model calibrated to live plant state without the solver gaming them.
# NOTE: BFW_Imbalance is intentionally NOT in this set — it is kept as
# m.Var() bounded to +/-20 t/hr so the steam-cycle mass balance has room
# to close (see cell 4 "Production-Grade Process Limits").
IMBALANCE_PARAMS = {
    "HP_Steam_Imbalance", "MP_Steam_Imbalance", "LP_Steam_Imbalance",
    "Total_Fuel_Imbalance", "BFW_Imbalance_Enthalpy",
    "BFW_A_Motor_Power_Imbalance", "BFW_B_Turb_Steam_Imbalance",
    "BFW_C_Turb_Steam_Imbalance", "BFW_D_Motor_Power_Imbalance",
    "BFW_E_Turb_Steam_Imbalance", "BFW_F_Motor_Power_Imbalance",
    "DMW_Turbine_A_Steam_Imbalance", "DMW_Turbine_C_Steam_Imbalance",
    "DMW_Pump_Motor_B_Power_Imbalance",
    "Air_Compressor_Turbine_A_Steam_Imbalance", "Air_Compressor_Turbine_D_Steam_Imbalance",
    "Air_Compressor_Motor_C_Power_Imbalance", "Air_Compressor_Motor_B_Power_Imbalance",
}

# Empty by default — every row in constraints sheet is enforced.
SKIP_CONSTRAINTS = set()

# Derived equations that reference *_Imbalance tags (handled as parameters)
SKIP_DERIVED_IMBALANCE = set()

# -- PYTHON IMPORTS --------------------------------------------------------
import re, math, logging, warnings, os
import numpy as np
import pandas as pd
import networkx as nx
from datetime import datetime
from typing import Dict, List, Optional, Set, Tuple, Any
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Imports OK. Feature file:", FEATURE_FILE)


Imports OK. Feature file: feature_file_eo_v7_unified.xlsx


In [2]:
TAG_PATTERN = re.compile(r"\[([^\[\]]+)\]")

def extract_tag_refs(formula):
    if not isinstance(formula, str): return []
    return [t.strip().replace(" ", "_") for t in TAG_PATTERN.findall(formula)]

def preprocess_formula(formula):
    if not isinstance(formula, str): return str(formula)
    formula = formula.replace("(Total_Fuel_Consumption_U_O)", "([Total_Fuel_Consumption_U_O])")
    s = TAG_PATTERN.sub(lambda m: m.group(1).strip().replace(" ", "_"), formula)
    s = s.replace("^", "**").replace("&&", " and ").replace("||", " or ")
    # FIX 1: Case-insensitive match, allows spaces like "IF ("
    s = re.sub(r"(?i)\bif\s*\(", "if_(", s)
    return s

def _make_eval_env(context, all_tag_names=None):
    def _if_(c, t, f):
        try:
            cond = float(c)
            # FIX 2: Prevent np.nan from evaluating to True
            if np.isnan(cond):
                cond = 0.0
        except:
            cond = bool(c)
        return t if cond else f

    def _min_(*a):
        args = list(a[0]) if len(a)==1 and hasattr(a[0],"__iter__") else list(a)
        v = [float(x) for x in args if not (isinstance(x,float) and np.isnan(x))]
        return min(v) if v else np.nan
    def _max_(*a):
        args = list(a[0]) if len(a)==1 and hasattr(a[0],"__iter__") else list(a)
        v = [float(x) for x in args if not (isinstance(x,float) and np.isnan(x))]
        return max(v) if v else np.nan
    def _avg_(*a):
        args = list(a[0]) if len(a)==1 and hasattr(a[0],"__iter__") else list(a)
        v = [float(x) for x in args if not (isinstance(x,float) and np.isnan(x))]
        return sum(v)/len(v) if v else np.nan
    def _sum_(*a):
        args = list(a[0]) if len(a)==1 and hasattr(a[0],"__iter__") else list(a)
        v = [float(x) for x in args if x is not None and not (isinstance(x,float) and np.isnan(x))]
        return sum(v) if v else 0.0
    def _missing(x):
        if x is None: return 1.0
        if isinstance(x,float) and np.isnan(x): return 1.0
        return 0.0
    def _safe_sqrt(x):
        try: return math.sqrt(max(float(x),0))
        except: return np.nan
    def _safe_log(x):
        try: return math.log(max(float(x),1e-30))
        except: return np.nan

    env = {"__builtins__":{}, "nan":np.nan, "inf":np.inf, "pi":math.pi,
           "if_":_if_, "if":_if_, "If":_if_,
           "min":_min_, "max":_max_, "avg":_avg_, "sum":_sum_,
           "abs":abs, "round":round,
           "sqrt":_safe_sqrt, "log":_safe_log, "exp":math.exp, "ln":_safe_log,
           "ceil":math.ceil, "floor":math.floor, "trunc":math.trunc,  # [PATCH-EVAL-v1]
           "sin":math.sin, "cos":math.cos,
           "missing":_missing, "MISSING_NUMERIC":float("nan"),
           "_safe_div": (lambda a,b: float("nan") if (b==0 or b is None) else a/b),
           "True":True, "False":False}  # [PATCH-EVAL-v1]

    if all_tag_names:
        for tn in all_tag_names:
            if tn not in env: env[tn] = np.nan

    env.update({k:v for k,v in context.items() if not (isinstance(v,str) or callable(v))})
    return env

def safe_eval_scalar(formula, context, all_tags=None):
    try:
        return float(eval(preprocess_formula(formula), _make_eval_env(context, all_tags)))
    except Exception:
        return np.nan

def topo_sort_formulas(formula_map):
    G = nx.DiGraph()
    all_tags = set(formula_map.keys())
    for tag, formula in formula_map.items():
        G.add_node(tag)
        for ref in extract_tag_refs(formula):
            if ref in all_tags and ref != tag:
                G.add_edge(ref, tag)
    cycles = list(nx.simple_cycles(G))
    for cycle in cycles:
        for i in range(len(cycle)):
            try: G.remove_edge(cycle[i], cycle[(i+1)%len(cycle)])
            except nx.NetworkXError: pass
    return list(nx.topological_sort(G)), cycles

print("Formula engine ready.")

Formula engine ready.


In [3]:
print("="*70); print("TASK 1 — DATA INGESTION"); print("="*70)
xl = pd.ExcelFile(FEATURE_FILE)
cfg = {}
for sheet in ["tag","inferred","variables","derived_equations","constraints","objective","model_parameter","master_pi_data"]:
    if sheet in xl.sheet_names:
        cfg[sheet] = xl.parse(sheet).dropna(how="all").reset_index(drop=True)
        print(f"  {sheet:<25} : {len(cfg[sheet]):>5} rows")
    else:
        cfg[sheet] = pd.DataFrame()
        print(f"  {sheet:<25} : MISSING")

model_params = {}
for _,row in cfg["model_parameter"].iterrows():
    p = str(row.get("parameter","")).strip(); v = row.get("value",np.nan)
    if p and p != "nan" and pd.notna(v):
        try: model_params[p] = float(v)
        except: pass
print(f"\n  Model parameters loaded: {len(model_params)}")

print("\n" + "="*70); print("TASK 2 — PI SNAPSHOT + WORKING CONTEXT"); print("="*70)
df_pi = cfg["master_pi_data"].copy()
ts_col = next((c for c in df_pi.columns if "date" in c.lower()), None)
if ts_col: df_pi[ts_col] = pd.to_datetime(df_pi[ts_col], errors="coerce")

if PI_ROW_SELECTION == "best":
    snapshot_idx = df_pi.notna().sum(axis=1).idxmax()
elif PI_ROW_SELECTION == "first":
    snapshot_idx = 0
elif PI_ROW_SELECTION == "last":
    snapshot_idx = len(df_pi) - 1
else:
    snapshot_idx = int(PI_ROW_SELECTION)
pi_snapshot = df_pi.iloc[[snapshot_idx]].reset_index(drop=True)
ts_value = pi_snapshot[ts_col].iloc[0] if ts_col else "N/A"

def build_working_ctx_from_row(row, model_params):
    ctx = {}
    for col in row.index:
        if ts_col and col == ts_col: continue
        v = row[col]
        if pd.notna(v):
            try: ctx[col] = float(v)
            except: pass
    ctx.update(model_params)
    ctx.setdefault("Whatif_running", 0.0)
    ctx.setdefault("opt_flag", 0.0)
    ctx.setdefault("act_running", 1.0)
    PI_ALIAS = {
        "AC_A_Flow_To_Header_raw":"Air_compressor_A_Discharge_flow_raw",
        "AC_B_Flow_To_Header_raw":"Air_compressor_B_Discharge_flow_raw",
        "AC_C_Flow_To_Header_raw":"Air_compressor_C_Discharge_flow_raw",
        "AC_D_Flow_To_Header_raw":"Air_compressor_D_Discharge_flow_raw",
    }
    for a,rc in PI_ALIAS.items():
        if a not in ctx and rc in ctx: ctx[a] = ctx[rc]
    lhv = ctx.get("LHV_Raw", np.nan)
    if not (isinstance(lhv,float) and np.isnan(lhv)):
        for t in ["LHV_Raw_2","LHV_Raw_3","LHV_Raw_4","LHV_Raw_5"]:
            ctx.setdefault(t, lhv)
    return ctx

working_ctx = build_working_ctx_from_row(pi_snapshot.iloc[0], model_params)
print(f"  Snapshot: {ts_value} (row idx {snapshot_idx})")
print(f"  PI values populated: {len(working_ctx)}")

TASK 1 — DATA INGESTION


  tag                       :  4497 rows
  inferred                  :  3342 rows
  variables                 :   172 rows
  derived_equations         :   119 rows
  constraints               :    44 rows
  objective                 :     1 rows
  model_parameter           :     9 rows
  master_pi_data            :     1 rows

  Model parameters loaded: 9

TASK 2 — PI SNAPSHOT + WORKING CONTEXT
  Snapshot: 2026-03-31 00:00:00 (row idx 0)
  PI values populated: 1485


In [4]:
print("="*70); print("TASK 3 — INFERRED CHAIN (snapshot)"); print("="*70)

inferred_formula_map = {}
for _,row in cfg["inferred"].iterrows():
    tag = str(row.get("tag_name","")).strip()
    f = row.get("formula_expression","")
    if tag and tag != "nan":
        inferred_formula_map[tag] = str(f) if pd.notna(f) else ""
if "Fuel_Bill" in inferred_formula_map:
    inferred_formula_map["Fuel_Bill"] = inferred_formula_map["Fuel_Bill"].replace(
        "(Total_Fuel_Consumption_U_O)", "([Total_Fuel_Consumption_U_O])")

all_referenced_tags = set()
for f in inferred_formula_map.values(): all_referenced_tags.update(extract_tag_refs(f))
for _,r in cfg["derived_equations"].iterrows():
    f = r.get("formula_expression","")
    if isinstance(f,str): all_referenced_tags.update(extract_tag_refs(f))
for _,r in cfg["constraints"].iterrows():
    f = r.get("expression","")
    if isinstance(f,str): all_referenced_tags.update(extract_tag_refs(f))
all_referenced_tags.update(inferred_formula_map.keys())
all_referenced_tags.update(c for c in df_pi.columns if c != ts_col)

sorted_inf, circular_inf = topo_sort_formulas(inferred_formula_map)
if circular_inf: print(f"  Circular dependencies removed: {len(circular_inf)}")

n_computed = n_skipped = 0
for tag in sorted_inf:
    f = inferred_formula_map.get(tag,"")
    if not f.strip(): continue
    v = safe_eval_scalar(f, working_ctx, all_referenced_tags)
    if not np.isnan(v):
        working_ctx[tag] = v; n_computed += 1
    else:
        n_skipped += 1

print(f"  Inferred formulas: {len(inferred_formula_map)}  Computed: {n_computed}  Skipped (NaN): {n_skipped}")
print(f"  Total tags referenced anywhere: {len(all_referenced_tags)}")

# Mass-balance verification
print("\n  Mass-balance check at snapshot (residuals should be ~0):")
# FIX: Peek at derived equations just for the mass balance check to avoid NaN errors
derived_map = {str(r.get("tag_name","")).strip(): r.get("formula_expression","") for _,r in cfg["derived_equations"].iterrows()}

def chk(name, gen, imb, cons):
    # Evaluate on the fly if they are in derived_equations and not yet in working_ctx
    for tag in [gen, imb, cons]:
        if tag not in working_ctx and tag in derived_map:
            working_ctx[tag] = safe_eval_scalar(derived_map[tag], working_ctx, all_referenced_tags)

    g = working_ctx.get(gen, np.nan); i = working_ctx.get(imb, np.nan); c = working_ctx.get(cons, np.nan)
    if all(not (isinstance(x,float) and np.isnan(x)) for x in [g,i,c]):
        r = g - (i+c)
        flag = "✓" if abs(r) < 1e-3 else "⚠"
        print(f"    {flag} {name:<10}  Gen={g:9.3f}  Imb={i:9.3f}  Cons={c:9.3f}  Residual={r:+.4f}")
    else:
        print(f"    ? {name:<10}  Gen={g}  Imb={i}  Cons={c}  → NaN, cannot verify")

chk("HP Steam","Total_BLR_HPS_Generation","HP_Steam_Imbalance","HP_Steam_Consumption")
chk("MP Steam","MP_Steam_Generation","MP_Steam_Imbalance","MP_Steam_Consumption")
chk("LP Steam","LP_Steam_Generation","LP_Steam_Imbalance","LP_Steam_Consumption")
chk("BFW",     "Total_BFW_Generated","BFW_Imbalance","Total_BFW_Consumption")

TASK 3 — INFERRED CHAIN (snapshot)


  Circular dependencies removed: 13


  Inferred formulas: 3342  Computed: 3049  Skipped (NaN): 293
  Total tags referenced anywhere: 5433

  Mass-balance check at snapshot (residuals should be ~0):
    ✓ HP Steam    Gen=  313.201  Imb=    8.658  Cons=  304.543  Residual=+0.0000
    ✓ MP Steam    Gen=   81.301  Imb=   10.000  Cons=   71.301  Residual=+0.0000
    ✓ LP Steam    Gen=  235.326  Imb=   10.298  Cons=  225.028  Residual=+0.0000
    ✓ BFW         Gen= 1055.136  Imb=   18.672  Cons= 1036.464  Residual=+0.0000


In [5]:
import re
import pandas as pd
import numpy as np

print("="*70); print("TASK 4 — SUFFIXING & FORMULA PRE-PROCESSING"); print("="*70)

# --- PATCH: Update preprocess_formula to handle ||, &&, and if() ---
TAG_PATTERN = re.compile(r"\[([^\[\]]+)\]")

def preprocess_formula(formula):
    if not isinstance(formula, str): return str(formula)
    formula = formula.replace("(Total_Fuel_Consumption_U_O)", "([Total_Fuel_Consumption_U_O])")

    # 1. Replace Tags
    s = TAG_PATTERN.sub(lambda m: m.group(1).strip().replace(" ", "_"), formula)

    # 2. Math operators
    s = s.replace("^", "**")

    # 3. Logical operators (Excel/C to Python native)
    s = s.replace("||", " or ")
    s = s.replace("&&", " and ")

    # 4. FIX: Rename 'if(' to 'if_(' to avoid Python reserved keyword SyntaxError
    s = re.sub(r'(?i)\bif\s*\(', 'if_(', s)

    # 5. [PATCH-EVAL-v1] — MISSING_NUMERIC sentinel (DB parity)
    s = s.replace('MISSING_NUMERIC', 'MISSING_NUMERIC')  # symbol comes from eval env

    return s
# ---------------------------------------------------------------------

pi_tag_names = set(cfg["tag"][cfg["tag"]["tag_type"]=="pi"]["tag_name"].tolist())
actual_ns = dict(working_ctx)
for tag in pi_tag_names:
    if tag in working_ctx: actual_ns[f"{tag}_actual"] = working_ctx[tag]
for tag in inferred_formula_map:
    if tag in working_ctx: actual_ns[f"{tag}_actual"] = working_ctx[tag]
print(f"  actual_ns size: {len(actual_ns)}")


print("\n" + "="*70); print("TASK 5 — VARIABLE BOUNDS (DYNAMIC EXPRESSIONS)"); print("="*70)

# Create the evaluation environment once using the live snapshot data
env_pure = _make_eval_env(actual_ns, all_referenced_tags)

# Helper function to evaluate dynamic strings to floats
def evaluate_dynamic_bound(expr_raw, default_val=None):
    if pd.isna(expr_raw) or str(expr_raw).strip() == "" or str(expr_raw).strip().lower() == "nan":
        return default_val
    expr_str = str(expr_raw).strip()
    try:
        # If it is already a flat number, return it immediately
        return float(expr_str)
    except ValueError:
        # It is a formula/expression containing tags. Preprocess and evaluate.
        try:
            prep = preprocess_formula(expr_str)
            return float(eval(prep, env_pure))
        except Exception as e:
            # Fallback if a specific bound fails to parse
            print(f"    [Warning] Failed to eval bound '{expr_str[:30]}...': {e}")
            return default_val

var_defs = {}
imbalance_param_vals = {}
n_missing_pi = 0; n_imbalance_as_param = 0

for _,row in cfg["variables"].iterrows():
    vname = str(row.get("tag_name","")).strip()
    if not vname or vname == "nan": continue

    if vname in IMBALANCE_PARAMS:
        v = actual_ns.get(vname, np.nan)
        imbalance_param_vals[vname] = float(v) if not (isinstance(v,float) and np.isnan(v)) else 0.0
        n_imbalance_as_param += 1
        continue

    # 1. Get Physical Bounds (Absolute Limits)
    lb_phys_raw = row.get("lower_bound_value")
    ub_phys_raw = row.get("upper_bound_value")

    lb_phys = float(lb_phys_raw) if pd.notna(lb_phys_raw) and str(lb_phys_raw).strip() != "" else 0.0
    ub_phys = float(ub_phys_raw) if pd.notna(ub_phys_raw) and str(ub_phys_raw).strip() != "" else max((lb_phys or 0)*2, 1e6)
    if lb_phys > ub_phys: lb_phys, ub_phys = ub_phys, lb_phys

    # 2. Get Scenario Bounds (Dynamic Evaluated Limits)
    lb_raw = row.get("lower_bound_expression")
    ub_raw = row.get("upper_bound_expression")

    lb_scen = evaluate_dynamic_bound(lb_raw, default_val=lb_phys)
    ub_scen = evaluate_dynamic_bound(ub_raw, default_val=ub_phys)
    if lb_scen > ub_scen: lb_scen, ub_scen = ub_scen, lb_scen

    # 3. Initialization: Use purely live PI data for the warmest start
    raw_val = actual_ns.get(vname, np.nan)

    if isinstance(raw_val, (int, float)) and np.isfinite(raw_val):
        init = float(raw_val)
        init_orig = float(raw_val)
    else:
        # Fallback only if tag is completely missing or NaN
        init = float(lb_phys)
        init_orig = float('nan')
        n_missing_pi += 1

    var_defs[vname] = {
        "lb_phys": lb_phys, "ub_phys": ub_phys,
        "lb_scen": lb_scen, "ub_scen": ub_scen,
        "init": init,
        "is_integer": bool(row.get("flag_integer", 0) == 1),
        "init_orig": init_orig
    }




# =========================================================================
# [FIX-1] INTEGER-FLAG SANITATION (python-only override)
# -------------------------------------------------------------------------
# Problem observed in feature_file_eo_v6.xlsx `variables` sheet:
#   * Continuous flow/power vars are mistakenly flagged integer=1, e.g.:
#       BLR_1_HPS_Gen, BFW_C_Turb_Steam, FD_Fan_BLR_1_Steam,
#       BFW_Flow_Consumption, Air_Compressor_Motor_B_Power, ...
#   * Some genuine status bits are NOT flagged integer, e.g.:
#       BFW_B_Turb_Status, VHP_BFW_*_Status
#   * Several rows are duplicated with inconsistent flags.
#   -> With 44 integer vars (many nonsense), APOPT branch-and-bound
#      explodes and returns fractional relaxations (status=0.297 etc.).
#
# Python fix:
#   * Any var ending in "_Status" is forced integer with bounds [0,1].
#   * Named counters (BFW_Drives_Running etc.) kept integer.
#   * All other vars forced continuous. lb clamped to 0 on counters so an
#     "all off" configuration is reachable.
#
# EXCEL FIX: in `variables` sheet, set flag_integer = 1 ONLY on *_Status
# rows and on the counter rows listed in _INT_ALLOW_NAMES. Remove duplicates.
# =========================================================================
_INT_ALLOW_SUFFIX = ("_Status",)
_INT_ALLOW_NAMES = {
    "BFW_Drives_Running","BFW_Motors_Running","BFW_Turbines_Running",
    "VHP_BFW_Drives_Running","VHP_BFW_Motors_Running","VHP_BFW_Turbines_Running",
    "CW_Motors_Running","CW_Turbines_Running","CW_Drives_Running",
    "DMW_Drives_Running","DMW_Motors_Running","DMW_Turbines_Running",
    "Air_Compressor_Drives_Running","Air_Compressor_Motors_Running","Air_Compressor_Turbines_Running",
    "Total_Boilers_Running",
}
_n_demoted = 0; _n_promoted = 0; _n_clamped = 0
for _vn, _vd in var_defs.items():
    should_be_int = _vn.endswith(_INT_ALLOW_SUFFIX) or _vn in _INT_ALLOW_NAMES
    if _vd["is_integer"] and not should_be_int:
        _vd["is_integer"] = False
        _n_demoted += 1
    elif should_be_int and not _vd["is_integer"]:
        _vd["is_integer"] = True
        _n_promoted += 1
    # status bits are strictly {0,1}; clamp UB to 1
    if should_be_int and _vn.endswith("_Status"):
        if _vd["ub_phys"] > 1 or _vd["ub_scen"] > 1:
            _vd["ub_phys"] = 1.0; _vd["ub_scen"] = 1.0; _n_clamped += 1
        _vd["lb_phys"] = 0.0; _vd["lb_scen"] = 0.0
# Counters: allow "all off" (lb=0)
for _vn in _INT_ALLOW_NAMES:
    if _vn in var_defs:
        var_defs[_vn]["lb_phys"] = 0.0
        if var_defs[_vn]["lb_scen"] > 0: var_defs[_vn]["lb_scen"] = 0.0

print(f"  [FIX-1] Integer-flag sanitation: demoted={_n_demoted} "
      f"promoted={_n_promoted} ub-clamped={_n_clamped}")
print(f"  [FIX-1] True integers retained   : "
      f"{sum(1 for _v in var_defs.values() if _v['is_integer'])}")

print(f"  Decision variables: {len(var_defs)}")
print(f"  Imbalance tags moved to m.Param: {n_imbalance_as_param}")
print(f"  Variables missing PI data (defaulted to Physical LB): {n_missing_pi}")
if n_missing_pi > 0:
    print("\n  Missing Variables:")
    for v,d in var_defs.items():
        if pd.isna(d['init_orig']):
            print(f"    {v:<55} PI=      NaN  →  start={d['init']:>10.4f}  phys_bounds=[{d['lb_phys']:.4f}, {d['ub_phys']:.4f}]")


print("\n" + "="*70); print("TASK 6 — PRE-SOLVE CONSTRAINT DIAGNOSTIC"); print("="*70)

def _strip_parens(s):
    s = s.strip()
    if s.startswith("(") and s.endswith(")"):
        d = 0
        for i,ch in enumerate(s):
            if ch == "(": d += 1
            elif ch == ")": d -= 1
            if d == 0 and i < len(s)-1: return s
        return s[1:-1]
    return s

print(f"  {'#':<3} {'System':<22} {'Op':<3} {'LHS':>14} {'RHS':>14} {'Residual':>12}  Status")
print("  " + "-"*82)
diag_rows = []; start_violations = 0; start_ok = 0
for i,r in cfg["constraints"].iterrows():
    if i in SKIP_CONSTRAINTS:
        start_violations += 1  # count as NaN
        diag_rows.append({"#":i,"System":str(r.get("system","")),"Expression":str(r.get("expression","")),"Op":"==","LHS":float("nan"),"RHS":float("nan"),"Residual":float("nan"),"Status":"SKIPPED (enthalpy)"})
        continue
    raw = str(r.get("expression","")).strip()
    sys = str(r.get("system",""))
    if not raw or raw == "nan": continue
    s = _strip_parens(raw)
    prep = preprocess_formula(s)
    op = None; lhs_s = rhs_s = None
    for o in ["==",">=","<=",">","<"]:
        if o in prep: op = o; lhs_s, rhs_s = prep.split(o,1); break
    if not op: continue

    # env_pure is already created in Task 5 and contains updated namespace
    try: lhs = float(eval(lhs_s.strip(), env_pure))
    except: lhs = float("nan")
    try: rhs = float(eval(rhs_s.strip(), env_pure))
    except: rhs = float("nan")

    if np.isnan(lhs) or np.isnan(rhs):
        status = "NaN"; resid = np.nan; start_violations += 1
    elif op == "==":
        resid = lhs - rhs
        if abs(resid) > 1e-2: status = "VIOLATED"; start_violations += 1
        else: status = "OK"; start_ok += 1
    elif op in (">=",">"):
        resid = max(rhs-lhs, 0)
        if lhs < rhs - 1e-2: status = "VIOLATED"; start_violations += 1
        else: status = "OK"; start_ok += 1
    else:
        resid = max(lhs-rhs, 0)
        if lhs > rhs + 1e-2: status = "VIOLATED"; start_violations += 1
        else: status = "OK"; start_ok += 1

    lhs2 = f"{lhs:14.4f}" if not np.isnan(lhs) else "         NaN  "
    rhs2 = f"{rhs:14.4f}" if not np.isnan(rhs) else "         NaN  "
    res2 = f"{resid:12.4f}" if not np.isnan(resid) and resid != 0 else "          - "
    print(f"  {i:<3} {sys[:22]:<22} {op:<3} {lhs2} {rhs2} {res2}  {status}")
    diag_rows.append({"#":i,"System":sys,"Expression":raw,"Op":op,"LHS":lhs,"RHS":rhs,
                      "Residual":resid,"Status":status})
print("  " + "-"*82)
print(f"  RESULT: {start_ok} OK, {start_violations} violated/NaN  →  {'GO' if start_violations==0 else 'INSPECT BEFORE SOLVING'}")

pre_solve_diag_df = pd.DataFrame(diag_rows)


print("\n" + "=" * 70)
print("TASK 5b - QC OVERRIDES (Python-only, Excel unchanged)")
print("=" * 70)

# (A) Integer counter upper bounds - sheet has ub=1 for counters whose
#     derived formula can sum up to more than one status. Override here.
INTEGER_COUNTER_OVERRIDES = {
    'BFW_Drives_Running':              (0, 6),  # A+B+C+D+E+F
    'CW_Motors_Running':               (0, 4),  # C+D+E+F
    'CW_Turbines_Running':             (0, 3),  # A+B+G
    'DMW_Turbines_Running':            (0, 2),  # A+C
    'Air_Compressor_Motors_Running':   (0, 2),  # B+C
    'Air_Compressor_Turbines_Running': (0, 2),  # A+D
    'VHP_BFW_Turbines_Running':        (0, 2),  # B+C
}
n_int_fixed = 0
for vname, (lo, hi) in INTEGER_COUNTER_OVERRIDES.items():
    if vname in var_defs:
        vd = var_defs[vname]
        vd['lb_phys'] = float(lo); vd['ub_phys'] = float(hi)
        vd['lb_scen'] = float(lo); vd['ub_scen'] = float(hi)
        init = vd.get('init', lo)
        try: init = float(init)
        except: init = float(lo)
        if not np.isfinite(init): init = float(lo)
        vd['init'] = int(max(lo, min(hi, round(init))))
        n_int_fixed += 1
print(f"  Integer counter bounds overridden: {n_int_fixed}")

# (B) Continuous ceilings - widen physical UB to guarantee Stage 1
#     feasibility (live plant may exceed Excel ub_value). Scenario
#     bounds widened only if live value already exceeds them.
WIDE_CEILING = 5000.0
n_cont_widened = 0; n_scen_patched = 0
for vname, vd in var_defs.items():
    if vd.get('is_integer'): continue
    if vd['ub_phys'] < WIDE_CEILING:
        vd['ub_phys'] = WIDE_CEILING; n_cont_widened += 1
    if vd['lb_phys'] < 0: vd['lb_phys'] = 0.0
    live = vd.get('init', None)
    try: live_f = float(live)
    except: live_f = float('nan')
    if np.isfinite(live_f):
        if vd['ub_scen'] < live_f:
            vd['ub_scen'] = max(vd['ub_scen'], live_f * 1.1, live_f + 1.0); n_scen_patched += 1
        if vd['lb_scen'] > live_f:
            vd['lb_scen'] = min(vd['lb_scen'], live_f); n_scen_patched += 1
print(f"  Continuous ceilings widened to {WIDE_CEILING}: {n_cont_widened}")
print(f"  Scenario bounds patched to contain live value: {n_scen_patched}")

# (C) Sanity: lb <= ub after all overrides
for vname, vd in var_defs.items():
    if vd['lb_phys'] > vd['ub_phys']:
        vd['lb_phys'], vd['ub_phys'] = vd['ub_phys'], vd['lb_phys']
    if vd['lb_scen'] > vd['ub_scen']:
        vd['lb_scen'], vd['ub_scen'] = vd['ub_scen'], vd['lb_scen']
    try: init_f = float(vd['init'])
    except: init_f = vd['lb_phys']
    if not np.isfinite(init_f): init_f = vd['lb_phys']
    init_f = max(vd['lb_phys'], min(vd['ub_phys'], init_f))
    vd['init'] = int(round(init_f)) if vd.get('is_integer') else init_f
print("  Sanity check complete: all bounds coherent.")

# ==============================================================================
# (D) PRODUCTION-GRADE OVERRIDES: Process Limits & Sensor Uncertainties
# ==============================================================================
print("\n  Applying Production-Grade Process Limits...")

# 1. Prevent DMW_Makeup from going to 0 (Blowdown/Vent physical losses)
if 'DMW_Makeup' in var_defs:
    var_defs['DMW_Makeup']['lb_phys'] = 220.0
    var_defs['DMW_Makeup']['lb_scen'] = 220.0
    if var_defs['DMW_Makeup']['init'] < 220.0:
        var_defs['DMW_Makeup']['init'] = 220.0
    print("    -> DMW_Makeup lower bound locked to 30.0 t/hr")

# 2. BFW_Imbalance bounds: Allow sensor drift (±20 t/hr), prevent massive hiding
if 'BFW_Imbalance' in var_defs:
    var_defs['BFW_Imbalance']['lb_phys'] = -20.0
    var_defs['BFW_Imbalance']['ub_phys'] = 20.0
    var_defs['BFW_Imbalance']['lb_scen'] = -20.0
    var_defs['BFW_Imbalance']['ub_scen'] = 20.0
    # Ensure starting value sits safely inside bounds
    curr_init = var_defs['BFW_Imbalance']['init']
    var_defs['BFW_Imbalance']['init'] = max(-20.0, min(20.0, curr_init))
    print("    -> BFW_Imbalance bounds restricted to [-20.0, 20.0] t/hr")

TASK 4 — SUFFIXING & FORMULA PRE-PROCESSING
  actual_ns size: 8427

TASK 5 — VARIABLE BOUNDS (DYNAMIC EXPRESSIONS)
    [Warning] Failed to eval bound 'aswqa...': name 'aswqa' is not defined
    [Warning] Failed to eval bound 'aqawsaweqa...': name 'aqawsaweqa' is not defined
  [FIX-1] Integer-flag sanitation: demoted=0 promoted=0 ub-clamped=0
  [FIX-1] True integers retained   : 44
  Decision variables: 156
  Imbalance tags moved to m.Param: 16
  Variables missing PI data (defaulted to Physical LB): 0

TASK 6 — PRE-SOLVE CONSTRAINT DIAGNOSTIC
  #   System                 Op             LHS            RHS     Residual  Status
  ----------------------------------------------------------------------------------
  0   Boilers                >=         62.7968        50.0000           -   OK
  1   Boilers                >=         65.7441        50.0000           -   OK
  2   HP Steam               ==        313.2008       313.2008           -   OK
  3   MP Steam               ==         81.

In [6]:
pip install gekko

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
print("="*70); print("TASK 7 — GEKKO MODEL BUILD"); print("="*70)
from gekko import GEKKO
import math

m = GEKKO(remote=False)
m.options.IMODE = 3
m.options.MAX_ITER = int(model_params.get("nlp_maximum_iterations", 1000))

# Variables (Built using wide PHYSICAL bounds for Stage 1)
gekko_vars = {}
for vname, vd in var_defs.items():
    lb_g = int(round(vd["lb_phys"])) if vd["is_integer"] else vd["lb_phys"]
    ub_g = int(round(vd["ub_phys"])) if vd["is_integer"] else vd["ub_phys"]
    gekko_vars[vname] = m.Var(value=vd["init"], lb=lb_g, ub=ub_g,
                              integer=vd["is_integer"], name=vname)
n_int = sum(1 for v in var_defs.values() if v["is_integer"])
print(f"  Decision Variables: {len(gekko_vars)} ({n_int} integer, {len(var_defs)-n_int} continuous)")

# Imbalance Parameters (frozen at PI values)
gekko_params = {}
for pname, pval in imbalance_param_vals.items():
    gekko_params[pname] = m.Param(value=pval, name=pname)
print(f"  Imbalance Parameters: {len(gekko_params)}")

def _is_scalar(x): return isinstance(x, (int,float,np.integer,np.floating))

def make_gekko_env(ctx, m_obj):
    def _if_(c, t, f):
        try: return t if float(c) else f
        except: return m_obj.if3(c, t, f)
    def _min_(*args):
        args = list(args[0]) if len(args)==1 and hasattr(args[0],"__iter__") else list(args)
        try: return min(float(a) for a in args)
        except:
            r = args[0]
            for a in args[1:]: r = m_obj.min2(r, a)
            return r
    def _max_(*args):
        args = list(args[0]) if len(args)==1 and hasattr(args[0],"__iter__") else list(args)
        try: return max(float(a) for a in args)
        except:
            r = args[0]
            for a in args[1:]: r = m_obj.max2(r, a)
            return r
    env = {"__builtins__":{}, "nan":0.0, "inf":1e30, "pi":math.pi,
           "if_":_if_, "if":_if_, "If":_if_, "min":_min_, "max":_max_,
           "avg":lambda *a: sum(float(x) for x in (a[0] if len(a)==1 else a))/len(a[0] if len(a)==1 else a),
           "abs":lambda x: m_obj.abs(x) if not _is_scalar(x) else abs(x),
           "sqrt":lambda x: m_obj.sqrt(x) if not _is_scalar(x) else math.sqrt(max(float(x),0)),
           "log":lambda x: m_obj.log(x) if not _is_scalar(x) else math.log(max(float(x),1e-30)),
           "ln":lambda x: m_obj.log(x) if not _is_scalar(x) else math.log(max(float(x),1e-30)),
           "exp":lambda x: m_obj.exp(x) if not _is_scalar(x) else math.exp(min(float(x),700)),
           "ceil":lambda x: math.ceil(float(x)) if _is_scalar(x) else x,
           "trunc":lambda x: math.trunc(float(x)) if _is_scalar(x) else x,
           "MISSING_NUMERIC": float("nan"),
           "_safe_div": (lambda a,b: float("nan") if (b==0 or b is None) else a/b),  # [PATCH-EVAL-v1]

           "floor":lambda x: math.floor(float(x)) if _is_scalar(x) else x,
           "round":round,
           "sum":lambda *a: sum(float(x) for x in (a[0] if len(a)==1 else a)),
           "sin":math.sin, "cos":math.cos, "missing":lambda x: 0.0,
           "True":True, "False":False, "m":m_obj}
    for t in all_referenced_tags:
        if t not in env: env[t] = 0.0
    env.update(ctx)
    return env

gekko_ctx = {}
for tag, val in actual_ns.items():
    if tag.endswith("_actual"): continue
    gekko_ctx[tag] = 0.0 if (isinstance(val,float) and np.isnan(val)) else val
gekko_ctx.update(gekko_vars)
gekko_ctx.update(gekko_params)
gekko_env = make_gekko_env(gekko_ctx, m)

# Derived equations → m.Intermediate (or m.Equation if tag is also a Var)
derived_eq_map = {}
for _,row in cfg["derived_equations"].iterrows():
    tag = str(row.get("tag_name","")).strip()
    frm = row.get("formula_expression","")
    if tag and tag != "nan" and isinstance(frm,str) and frm.strip():
        derived_eq_map[tag] = frm
sorted_derived, _ = topo_sort_formulas(derived_eq_map)

gekko_interm = {}; failed_derived = []; n_eq_constraints = 0; n_skipped_derived = 0
for tag in sorted_derived:
    formula = derived_eq_map.get(tag,"")
    if not formula: continue
    if tag in imbalance_param_vals or tag in SKIP_DERIVED_IMBALANCE:
        n_skipped_derived += 1
        continue
    try:
        result = eval(preprocess_formula(formula), gekko_env)
        if tag in gekko_vars:
            m.Equation(gekko_vars[tag] == result); iv = gekko_vars[tag]; n_eq_constraints += 1
        else:
            iv = m.Intermediate(result, name=tag)
        gekko_interm[tag] = iv
        gekko_env[tag] = iv; gekko_ctx[tag] = iv
    except Exception as e:
        failed_derived.append((tag, str(e)))
print(f"  Intermediates: {len(gekko_interm)} ({n_eq_constraints} tied to DVs)  Skipped (Param already): {n_skipped_derived}  Failed: {len(failed_derived)}")

# Constraints
n_constraints = 0; failed_constraints = []
for ci,row in cfg["constraints"].iterrows():
    if ci in SKIP_CONSTRAINTS: continue
    raw = str(row.get("expression","")).strip()
    if not raw or raw == "nan": continue
    raw = _strip_parens(raw)
    prep = preprocess_formula(raw)
    try:
        for op in ["==",">=","<=",">","<"]:
            if op in prep:
                lhs_s, rhs_s = prep.split(op, 1)
                lhs = eval(lhs_s.strip(), gekko_env)
                rhs = eval(rhs_s.strip(), gekko_env)
                if op == "==": m.Equation(lhs == rhs)
                elif op in (">=",">"): m.Equation(lhs >= rhs)
                else: m.Equation(lhs <= rhs)
                n_constraints += 1; break
    except Exception as e:
        failed_constraints.append((raw[:60], str(e)))
print(f"  Constraints added: {n_constraints}  Failed: {len(failed_constraints)}")

# =========================================================================
# [FIX-2] PHYSICS REPAIR — status <-> flow link constraints (python-only)
# -------------------------------------------------------------------------
# Problem observed in feature_file_eo_v6.xlsx `constraints` sheet:
#   Row 46:  ([DMW_Turbine_A_Status]+[DMW_Pump_Motor_B_Status]+[DMW_Turbine_C_Status])
#              *(1-[Whatif_running]) >= (...)*(1-[Whatif_running])*0 / [DMW_Pump_Rated_Flow]
#            -> multiplied by *0 on RHS; reduces to 0 >= 0 (always true, dormant).
#   Row 47:  (status sum)*0 < (demand/rate)+1
#            -> LHS is 0; reduces to 0 < positive (always true, dormant).
#
# Consequence in previous runs:
#   DMW_Makeup could stay at 220 t/hr while ALL THREE DMW pump statuses = 0
#   -> physically impossible (no pump running can't deliver water).
#
# Python fix:
#   * DMW link:  DMW_Makeup <= sum(DMW_status) * DMW_Pump_Rated_Flow
#   * BFW availability:  sum(BFW drive statuses) >= 1
#     (total BFW generation > 0 mandates at least one pump running)
#
# EXCEL FIX: in `constraints` sheet, replace row 46 with
#   [DMW_Makeup] <= ([DMW_Turbine_A_Status]+[DMW_Pump_Motor_B_Status]
#                   +[DMW_Turbine_C_Status]) * [DMW_Pump_Rated_Flow]
# and add a new row:
#   ([BFW_A_Motor_Status]+[BFW_B_Turb_Status]+[BFW_C_Turb_Status]
#    +[BFW_D_Motor_Status]+[BFW_E_Turb_Status]+[BFW_F_Motor_Status]) >= 1
# Row 47 can be deleted (it is dead anyway).
# =========================================================================
_physics_added = 0
try:
    _dmw_sum = (gekko_vars["DMW_Turbine_A_Status"]
              + gekko_vars["DMW_Pump_Motor_B_Status"]
              + gekko_vars["DMW_Turbine_C_Status"])
    _rate = gekko_env.get("DMW_Pump_Rated_Flow", 350.0)
    if "DMW_Makeup" in gekko_vars:
        m.Equation(gekko_vars["DMW_Makeup"] <= _dmw_sum * float(_rate))
        _physics_added += 1
except Exception as _e:
    print(f"  [FIX-2] DMW link skipped: {_e}")

try:
    _bfw_sum = sum(gekko_vars[n] for n in [
        "BFW_A_Motor_Status","BFW_B_Turb_Status","BFW_C_Turb_Status",
        "BFW_D_Motor_Status","BFW_E_Turb_Status","BFW_F_Motor_Status"] if n in gekko_vars)
    if not isinstance(_bfw_sum, int):
        m.Equation(_bfw_sum >= 1); _physics_added += 1
except Exception as _e:
    print(f"  [FIX-2] BFW link skipped: {_e}")

print(f"  [FIX-2] Physics-repair constraints added: {_physics_added}")

# =========================================================================
# [FIX-3] COOLING-WATER STATUS LOCK  (python-only override)
# -------------------------------------------------------------------------
# Problem observed in feature_file_eo_v6.xlsx `constraints` sheet, row 18:
#   The CW-drives constraint was copy-pasted from the Air-Compressor
#   constraint and references Air_Compressor_*_Rated_Flow / Total_Air_Demand
#   instead of any CW quantity. There is therefore NO equation anywhere
#   in the model tying CW_*_Status to a cooling-water demand. The solver,
#   being rational, turns EVERY CW pump OFF to save power + steam.
#   In reality this would cause an immediate plant trip (loss of CW to
#   EG, LAO, ETH and utility coolers).
#
# Python fix (until Excel row 18 is corrected):
#   Freeze each CW status at its measured (actual) value from PI. This
#   removes the bogus savings channel without blocking the legitimate
#   BFW/DMW/Boiler/Compressor optimization.
#
# Excel correction needed (permanent fix):
#   Rewrite constraint row 18 to enforce:
#       sum_i( CW_Pump_i_Rated_Flow * CW_i_Status ) >= Total_CW_Demand
#                                                      + Buffer_in_CW_Flow_Margin
#   and add `Total_CW_Demand` to derived_equations (sum of CW_TO_* tags).
#   Remove the erroneous `*0` factor and the Air-Compressor references.
# =========================================================================
_cw_locked = 0
_cw_names = ["CW_Motor_D_Status","CW_Motor_E_Status","CW_Motor_F_Status",
             "CW_Turbine_A_Status","CW_Turbine_B_Status","CW_Turbine_G_Status"]
for _n in _cw_names:
    if _n in gekko_vars and _n in var_defs:
        # Actual measured plant state: all 6 CW drives running. Hard-lock ON
        # until Excel constraint row 18 carries a real Total_CW_Demand balance.
        _act = 1
        # NOTE: Stage 3 rebuilds integer bounds from var_defs lb_phys/ub_phys,
        # so we must pin those too (otherwise MINLP releases the lock).
        var_defs[_n]["lb_phys"] = _act; var_defs[_n]["ub_phys"] = _act
        var_defs[_n]["lb_scen"] = _act; var_defs[_n]["ub_scen"] = _act
        var_defs[_n]["init"]    = _act
        try:
            gekko_vars[_n].lower = _act
            gekko_vars[_n].upper = _act
            _cw_locked += 1
        except Exception as _e:
            print(f"  [physics] CW lock skipped for {_n}: {_e}")
print(f"  CW status-lock constraints applied: {_cw_locked}")
# [/FIX-3]

# =========================================================================
# [FIX-4a] AIR-COMPRESSOR DISCHARGE UPPER BOUND  (python-only override)
# -------------------------------------------------------------------------
# Problem observed in feature_file_eo_v6.xlsx `constraints` sheet:
#   Rows 37-40 enforce only the LOWER bound:
#       Air_compressor_X_Discharge_flow >= X_Status * min_flow
#   There is NO upper bound of the form flow <= status * rated_max_flow.
#   Consequence: stopped units (status=0) can carry phantom discharge
#   flow (observed 0.136 t/hr on A, 0.74 on C) and running units can
#   over-discharge (B: 16.52, D: 20.11 vs Total_Air_Demand=25.9).
#
# Python fix (until Excel `constraints` sheet grows rows 37u-40u):
#   Add flow <= status * rated_max for each compressor. Rated max is
#   taken from the `variables` sheet upper_bound_value field:
#       A, B, D : 16.29 t/hr ;  C : 11.89 t/hr
# =========================================================================
_ac_pairs = [
    ("Air_compressor_A_Discharge_flow", "Air_Compressor_Turbine_A_Status", 16.29),
    ("Air_compressor_B_Discharge_flow", "Air_Compressor_Motor_B_Status",   16.29),
    ("Air_compressor_C_Discharge_flow", "Air_Compressor_Motor_C_Status",   11.89),
    ("Air_compressor_D_Discharge_flow", "Air_Compressor_Turbine_D_Status", 16.29),
]
_ac_added = 0
for _flow, _stat, _rmax in _ac_pairs:
    if _flow in gekko_vars and _stat in gekko_vars:
        try:
            m.Equation(gekko_vars[_flow] <= gekko_vars[_stat] * float(_rmax))
            _ac_added += 1
        except Exception as _e:
            print(f"  [FIX-4a] skipped {_flow}: {_e}")
print(f"  [FIX-4a] Air-comp discharge upper-bounds added: {_ac_added}")
# [/FIX-4a]

# =========================================================================
# [FIX-4b] COOLING-WATER TURBINE STEAM LOCK  (python-only override)
# -------------------------------------------------------------------------
# Problem observed: with CW statuses locked at 1 (FIX-3), the steam flow
# to each CW turbine is still a free variable in [0, 50]. There is no
# load-governing equation linking CW turbine steam to pump head/flow,
# so the solver uses CW_Turbine_B as a free LP-steam sink: 5.29 -> 36.1
# t/hr at the same duty. This is thermodynamically impossible and
# distorts the LP steam balance.
#
# Python fix (until `derived_equations` grows a proper steam-rate curve):
#   Pin each CW turbine steam to its measured (actual) value, same
#   approach as FIX-3 for statuses.
#
# Excel correction needed (permanent fix):
#   Add to derived_equations a steam-rate equation, e.g.:
#       CW_Turbine_X_Steam = k_X * CW_Turbine_X_Status * (CW_Pump_Head * CW_Flow_X)
#   or at minimum:
#       CW_Turbine_X_Steam <= CW_Turbine_X_Status * Design_Steam_X
# =========================================================================
# NOTE (investigation 2026-04-24): the earlier concern that CW_Turbine_B
# was a "free LP sink" jumping 5.29 -> 36.1 was WRONG. The infeasibility
# report from the capped run revealed that a derived equation in the
# Excel sheet already governs it:
#     CW_Turbine_B_Steam = CW_Turbine_B_Status * 38 * 0.95  (= 36.1)
# i.e. the formula forces each CW turbine to its design steam rate when
# ON, independent of actual pump load. The actual PI reading (5.29)
# simply diverges from this design formula (plant/design mismatch).
# No python override is therefore needed here; capping the variable
# creates infeasibility with the derived equation.
# Excel correction (permanent fix): rewrite derived_equations so
#   CW_Turbine_X_Steam = f(CW_pump_load_X)  rather than a fixed design rate.
# [/FIX-4b]


# =========================================================================

# =========================================================================


# =========================================================================

# =========================================================================


# =========================================================================

# =========================================================================


# =========================================================================

# =========================================================================



# =========================================================================


# =========================================================================


# ==============================================================================
# PRODUCTION-GRADE OVERRIDE: Lock Boiler Switchovers
# ==============================================================================
try:
    # Calculate how many boilers are currently running in the PI snapshot
    actual_boilers = sum(round(float(actual_ns.get(f"BLR_{i}_Status", 0))) for i in range(1, 6))

    # Grab the GEKKO variables for those 5 boilers
    blr_status_vars = [gekko_vars[f"BLR_{i}_Status"] for i in range(1, 6) if f"BLR_{i}_Status" in gekko_vars]

    # Enforce constraint: Sum of running boilers in optimization MUST EQUAL live count
    if len(blr_status_vars) == 5:
        m.Equation(m.sum(blr_status_vars) == actual_boilers)
        print(f"\n  [Production Lock] -> Forced exactly {actual_boilers} boilers to remain running.")
    else:
        print(f"\n  [Production Lock WARNING] -> Could not find all 5 boiler variables.")
except Exception as e:
    print(f"\n  [Production Lock WARNING] -> Failed to lock boiler count: {e}")
# ==============================================================================

# Objective
obj_row = cfg["objective"].iloc[0]
obj_tag = str(obj_row["tag_name"]).strip()
obj_dir = float(obj_row["direction"])
obj_formula = inferred_formula_map.get(obj_tag, "")
obj_expr = None
if obj_formula:
    try:
        obj_expr = eval(preprocess_formula(obj_formula), gekko_env)
        is_sym = not isinstance(obj_expr, (int, float, np.integer, np.floating))
        print(f"  Objective evaluated: symbolic={is_sym}")
        if not is_sym:
            print(f"  WARNING: Objective is constant {obj_expr} — solver cannot optimize!")
    except Exception as e:
        print(f"  WARNING: Objective eval failed: {e}")
if obj_expr is not None:
    if obj_dir == -1: m.Minimize(obj_expr)
    else: m.Maximize(obj_expr)
    print(f"  Objective: {obj_tag} ({'minimize' if obj_dir==-1 else 'maximize'})")


print("\n" + "="*70); print("TASK 7g — THREE-STAGE MINLP SOLVE PIPELINE"); print("="*70)

# ---------------------------------------------------------
# STAGE 1: Base Reconciliation (Wide Physical Bounds)
# ---------------------------------------------------------
print("  [Stage 1] Reconciling base plant with Physical Limits (IPOPT)...")
m.options.SOLVER = 3  # IPOPT
m.options.MAX_ITER = 500
try:
    m.solve(disp=False)
    print("  [Stage 1] SUCCESS: Stable base mass/energy balance found.")
except Exception as e:
    print(f"  [Stage 1] WARNING: Failed to find base physical state: {e}")

# ---------------------------------------------------------
# STAGE 2: Scenario Shift (Tight Scenario Bounds)
# ---------------------------------------------------------
print("\n  [Stage 2] Applying 'What-If' Scenario bounds and shifting loads (IPOPT)...")
n_tightened = 0
stage2_prev_bounds = {}
for vname, vd in var_defs.items():
    if vname not in gekko_vars: continue
    if vd["is_integer"]: continue  # keep integers at widened physical bounds until Stage 3
    if abs(vd["lb_scen"] - vd["lb_phys"]) > 1e-4 or abs(vd["ub_scen"] - vd["ub_phys"]) > 1e-4:
        stage2_prev_bounds[vname] = (gekko_vars[vname].lower, gekko_vars[vname].upper)
        gekko_vars[vname].lower = vd["lb_scen"]
        gekko_vars[vname].upper = vd["ub_scen"]
        n_tightened += 1

print(f"    -> Tightened {n_tightened} continuous variables to scenario limits (integers stay at physical).")
stage2_ok = False
try:
    m.solve(disp=False)
    stage2_ok = True
    print("  [Stage 2] SUCCESS: Continuous load shift completed.")
except Exception as e:
    print(f"  [Stage 2] WARNING: {e}  -  reverting continuous scenario bounds before Stage 3.")
    for vname,(lo,up) in stage2_prev_bounds.items():
        gekko_vars[vname].lower = lo
        gekko_vars[vname].upper = up

# ---------------------------------------------------------
# STAGE 3: Final Optimization & Integer Lock
# ---------------------------------------------------------
print("\n  [Stage 3] Locking Integer status constraints (APOPT)...")
# Before MINLP: revert continuous bounds to physical (wider) to give the
# branch-and-bound enough room to round integer statuses without violating
# mass/energy balance. Integers keep their widened physical bounds.
for vname, vd in var_defs.items():
    if vname not in gekko_vars: continue
    if vd["is_integer"]:
        gekko_vars[vname].lower = int(round(vd["lb_phys"]))
        gekko_vars[vname].upper = int(round(vd["ub_phys"]))
    else:
        gekko_vars[vname].lower = vd["lb_phys"]
        gekko_vars[vname].upper = vd["ub_phys"]

m.options.SOLVER = 1  # APOPT
m.solver_options = [
    'minlp_maximum_iterations 5000',
    'minlp_max_iter_with_int_sol 1000',
    'minlp_branch_method 3',
    'minlp_integer_tol 0.1',
    'minlp_gap_tol 0.01',
    'nlp_maximum_iterations 1000',
]
m.options.MAX_ITER = 5000

SOLVE_SUCCESS = False
try:
    m.solve(disp=True)
    SOLVE_SUCCESS = True
    print("\n  >>> MINLP SOLVER CONVERGED <<<")
except Exception as e:
    print(f"\n  >>> MINLP returned non-zero status ({e}) - attempting integer-rounding recovery <<<")

# RECOVERY: round integer vars to their current relaxation values, freeze them,
# then resolve as continuous NLP to restore mass/energy balance feasibility.
if not SOLVE_SUCCESS:
    def _gval(v):
        try:
            r = v.value
            return float(list(r)[0] if hasattr(r, "__iter__") else r)
        except Exception:
            return None
    n_rounded = 0
    for vname, vd in var_defs.items():
        if not vd["is_integer"]: continue
        if vname not in gekko_vars: continue
        cur = _gval(gekko_vars[vname])
        if cur is None: continue
        rounded = int(round(cur))
        rounded = max(int(round(vd["lb_phys"])), min(int(round(vd["ub_phys"])), rounded))
        gekko_vars[vname].lower = rounded
        gekko_vars[vname].upper = rounded
        n_rounded += 1
    print(f"  Recovery: rounded + fixed {n_rounded} integer DVs; resolving NLP...")
    # [FIX-5] Clear APOPT-only solver_options BEFORE switching to IPOPT,
    # otherwise IPOPT raises OPTION_INVALID on minlp_* keys and recovery dies.
    m.solver_options = []
    m.options.SOLVER = 3  # IPOPT
    m.options.MAX_ITER = 500
    try:
        m.solve(disp=False)
        SOLVE_SUCCESS = True
        print("  >>> RECOVERY NLP CONVERGED - using rounded-integer solution <<<")
    except Exception as e:
        print(f"  >>> Recovery NLP also failed: {e} <<<")
        from pathlib import Path
        infeas_path = Path(m._path) / "infeasibilities.txt"
        if infeas_path.exists():
            with open(infeas_path, 'r') as f:
                print("\n" + "="*70 + "\nINFEASIBILITIES REPORT\n" + "="*70)
                print(f.read()[:2000] + "\n... (truncated)")

print(f"\n  Solve result: {'SUCCESS' if SOLVE_SUCCESS else 'FAILED'}")

TASK 7 — GEKKO MODEL BUILD
  Decision Variables: 156 (44 integer, 112 continuous)
  Imbalance Parameters: 16
  Intermediates: 103 (102 tied to DVs)  Skipped (Param already): 16  Failed: 0
  Constraints added: 38  Failed: 0
  [FIX-2] Physics-repair constraints added: 2
  CW status-lock constraints applied: 6
  [FIX-4a] Air-comp discharge upper-bounds added: 4

  [Production Lock] -> Forced exactly 5 boilers to remain running.
  Objective evaluated: symbolic=True
  Objective: Objective_2 (minimize)

TASK 7g — THREE-STAGE MINLP SOLVE PIPELINE
  [Stage 1] Reconciling base plant with Physical Limits (IPOPT)...


  [Stage 1] SUCCESS: Stable base mass/energy balance found.

  [Stage 2] Applying 'What-If' Scenario bounds and shifting loads (IPOPT)...
    -> Tightened 110 continuous variables to scenario limits (integers stay at physical).


  [Stage 2] WARNING: @error: Solution Not Found
  -  reverting continuous scenario bounds before Stage 3.

  [Stage 3] Locking Integer status constraints (APOPT)...


 ----------------------------------------------------------------
 APMonitor, Version 1.0.3
 APMonitor Optimization Suite
 ----------------------------------------------------------------
 
 
 --------- APM Model Size ------------
 Each time step contains
   Objects      :  9
   Constants    :  0
   Variables    :  218
   Intermediates:  1
   Connections  :  30
   Equations    :  154
   Residuals    :  153
 
 Number of state variables:    218
 Number of total equations: -  169
 Number of slack variables: -  32
 ---------------------------------------
 Degrees of freedom       :    17
 
 Number of bound variables: -  6
 
 ----------------------------------------------
 Steady State Optimization with APOPT Solver
 ----------------------------------------------
Iter:     1 I:  0 Tm:      0.02 NLPi:    7 Dpth:    0 Lvs:    3 Obj:  4.73E+03 Gap:       NaN
Iter:     2 I: -1 Tm:      1.40 NLPi:  603 Dpth:    1 Lvs:    2 Obj:  4.73E+03 Gap:       NaN
Iter:     3 I:  0 Tm:      0.00 NLPi:    7 

In [8]:
from pathlib import Path
infeas_path = Path(m._path) / "infeasibilities.txt"
if infeas_path.exists():
    print("\n" + "="*70)
    print("INFEASIBILITIES REPORT")
    print("="*70)
    with open(infeas_path, 'r') as f:
        print(f.read())
else:
    print("\nNo infeasibilities.txt file found.")


INFEASIBILITIES REPORT
************************************************
***** POSSIBLE INFEASBILE EQUATIONS ************
************************************************
____________________________________________________________________________
EQ Number   Lower        Residual     Upper        Infeas.     Name
        25   0.0000E+00   1.0739E-03   0.0000E+00  -1.0739E-03  ss.sum_9.summation: 0 = y = \sum_{i=1}^{n} x_i
 Variable   Lower        Value        Upper        $Value      Name
       186  -1.2346E+20   4.9989E+00   1.2346E+20   0.0000E+00  ss.v170
       117   0.0000E+00   1.0000E+00   1.0000E+00   0.0000E+00  ss.int_blr_1_status
        19   0.0000E+00   1.0000E+00   1.0000E+00   0.0000E+00  ss.int_blr_2_status
        22   0.0000E+00   1.0000E+00   1.0000E+00   0.0000E+00  ss.int_blr_3_status
        25   0.0000E+00   1.0000E+00   1.0000E+00   0.0000E+00  ss.int_blr_4_status
        28   0.0000E+00   9.9785E-01   1.0000E+00   0.0000E+00  ss.int_blr_5_status
_____________

In [9]:
print("="*70); print("TASK 8 — RESULTS"); print("="*70)

def gval(v):
    try:
        raw = v.value
        return float(list(raw)[0] if hasattr(raw,"__iter__") else raw)
    except: return np.nan

opt_vals = {}
for vname, gv in gekko_vars.items():
    a = actual_ns.get(vname, np.nan)
    o = gval(gv)
    delta = (o - a) if not (np.isnan(a) or np.isnan(o)) else np.nan
    opt_vals[vname] = {"actual":a, "optimum":o, "delta":delta,
                       "is_integer":var_defs[vname]["is_integer"]}

opt_ctx = dict(actual_ns)
for vname, vals in opt_vals.items():
    if not np.isnan(vals["optimum"]): opt_ctx[vname] = vals["optimum"]

opt_derived = {}
for tag in sorted_derived:
    f = derived_eq_map.get(tag,"")
    if not f: continue
    val = safe_eval_scalar(f, opt_ctx, all_referenced_tags)
    if not np.isnan(val): opt_ctx[tag] = val; opt_derived[tag] = val

opt_inferred = {}
opt_inf_ctx = dict(opt_ctx)
for tag in sorted_inf:
    f = inferred_formula_map.get(tag,"")
    if not f.strip(): continue
    v = safe_eval_scalar(f, opt_inf_ctx, all_referenced_tags)
    if not np.isnan(v): opt_inf_ctx[tag] = v; opt_inferred[tag] = v

obj_actual = actual_ns.get(obj_tag, np.nan)
obj_optimum = opt_inf_ctx.get(obj_tag, np.nan)
n_changed = sum(1 for v in opt_vals.values() if not np.isnan(v["delta"]) and abs(v["delta"]) > 1e-10)

print(f"  Solve success: {SOLVE_SUCCESS}")
print(f"  {obj_tag} actual (Python):  {obj_actual:.6f}" if not np.isnan(obj_actual) else f"  {obj_tag} actual: N/A")
print(f"  {obj_tag} optimum (Python): {obj_optimum:.6f}" if not np.isnan(obj_optimum) else f"  {obj_tag} optimum: N/A")
print(f"  Variables changed: {n_changed} / {len(opt_vals)}")

# Show solver-reported objective (different from inferred re-evaluation)
solver_obj = np.nan  # FIX: Initialize to prevent NameError
if SOLVE_SUCCESS:
    try:
        solver_obj = m.options.OBJFCNVAL
        # GEKKO internal math flips the sign if maximizing
        if obj_dir == 1: solver_obj = -solver_obj

        print(f"\n  Solver-reported objective: {solver_obj:.4f}")
        print(f"  Baseline objective:        {obj_actual:.4f}")

        if not np.isnan(obj_actual) and obj_actual != 0:
            delta_s = solver_obj - obj_actual
            pct_s = delta_s / obj_actual * 100
            print(f"  Solver Δ Energy_Bill: {delta_s:+.4f} $/hr ({pct_s:+.2f}%)")
    except Exception as e:
        print(f"  Could not retrieve solver objective: {e}")

TASK 8 — RESULTS


  Solve success: True
  Objective_2 actual (Python):  4997.651837
  Objective_2 optimum (Python): 4997.651837
  Variables changed: 106 / 156

  Solver-reported objective: 4737.2193
  Baseline objective:        4997.6518
  Solver Δ Energy_Bill: -260.4325 $/hr (-5.21%)


In [10]:
# Provide safe fallbacks if multi-timestamp cell was skipped
try: inferred_all_df
except NameError: inferred_all_df = pd.DataFrame({"_placeholder":["multi-timestamp cell skipped"]})
try: completeness
except NameError:
    completeness = pd.Series(dtype=float, name="Pct_NonNull")

print("="*70); print("TASK 9 — EXPORT RESULTS"); print("="*70)

uom_map = {}
for _,row in cfg["tag"].iterrows():
    uom_map[str(row.get("tag_name","")).strip()] = str(row.get("data_type","")).strip()

# =========================================================================

# =========================================================================

# =========================================================================

# =========================================================================

# =========================================================================
# [FIX-4c] COST DISPLAY RECONCILIATION  (python-only override)
# -------------------------------------------------------------------------
# Problem observed in optimizer_eo_results_v01.xlsx:
#   Every cost line (Boiler_X_Fuel_Cost, Power_to_*, Sea_water_cost,
#   Costing_total_Fuel_Cost, etc.) shows IDENTICAL values in Actual_Data
#   and Optimized_Data columns, even though Total_Fuel_Consumption,
#   Fuel_BLR_X, and drive powers/steams changed. The `derived_equations`
#   formulas for these cost terms reference static design-point inputs
#   instead of the optimized decision variables, so the optimizer's
#   -$109/hr saving is invisible in the cost breakdown.
#
# Python fix (until `derived_equations` cost formulas are rewritten):
#   For each known cost-driver pair, recompute the optimized cost as
#       optimized_cost = actual_cost * (optimized_driver / actual_driver)
#   when the actual driver is non-zero. This preserves the per-unit
#   cost coefficient embedded in the baseline cost and scales it by
#   the optimized throughput.
#
# Excel correction needed (permanent fix):
#   Rewrite each cost formula in `derived_equations` to multiply the
#   decision-variable driver by a price constant, e.g.:
#       Boiler_X_Fuel_Cost = Fuel_BLR_X * Fuel_Price_per_MMBTU * heat_of_combustion
# =========================================================================
try:
    import numpy as _np
    # --- Per-boiler cost: proportional scaling on its own driver ------
    _prop_pairs = [
        ("Boiler_1_Fuel_Cost",      "Fuel_BLR_1"),
        ("Boiler_2_Fuel_Cost",      "Fuel_BLR_2"),
        ("Boiler_3_Fuel_Cost",      "Fuel_BLR_3"),
        ("Boiler_4_Fuel_Cost",      "Fuel_BLR_4"),
        ("Boiler_5_Fuel_Cost",      "Fuel_BLR_5"),
        ("Costing_total_Fuel_Cost", "Total_Fuel_Consumption"),
        ("DMW_Bill",                "DMW_Makeup"),
    ]
    def _getopt(_n):
        if _n in opt_vals: return opt_vals[_n]["optimum"]
        if _n in opt_derived: return opt_derived[_n]
        if _n in opt_inferred: return opt_inferred[_n]
        return None
    def _setopt(_n, _v):
        if   _n in opt_derived:  opt_derived[_n]  = _v
        elif _n in opt_inferred: opt_inferred[_n] = _v
        elif _n in opt_vals:     opt_vals[_n]["optimum"] = _v
        else: opt_inferred[_n]   = _v
    _n_fixed = 0
    for _cost_name, _drv_name in _prop_pairs:
        _a_cost = actual_ns.get(_cost_name); _a_drv = actual_ns.get(_drv_name); _o_drv = _getopt(_drv_name)
        if _a_cost is None or _a_drv is None or _o_drv is None: continue
        try: _a_cost=float(_a_cost); _a_drv=float(_a_drv); _o_drv=float(_o_drv)
        except Exception: continue
        if abs(_a_drv) < 1e-9 or _np.isnan(_a_cost): continue
        _new = _a_cost * (_o_drv / _a_drv)
        _setopt(_cost_name, _new); _n_fixed += 1
        print(f"    {_cost_name}: {_a_cost:.2f} -> {_new:.2f}  (driver {_drv_name}: {_a_drv:.4f} -> {_o_drv:.4f})")

    # --- Bill recomputation using OBJECTIVE-FUNCTION COEFFICIENTS -----
    # The GEKKO objective (solver log):
    #   Obj = (Total_Power_for_Drives + 45.0) * 3.6 * 13.334
    #       + (Total_Fuel_for_Boilers   + 0.4237) * 41.7425 * 2.04
    #       +  DMW_Makeup * 1.9573
    # So each bill MOVES by (delta_driver * effective_coefficient).
    # Fuel_Bill and Power_Bill display formulas in `inferred` use STATIC
    # "*_U_O" drivers that don't reflect optimizer decisions, so we
    # recompute here as: bill_opt = bill_actual + delta_driver * coef.
    _COEF_POWER = 3.6 * 13.334      # $/MW  (Total_Power_for_Drives)
    _COEF_FUEL  = 41.7425 * 2.04    # $/(t/hr fuel) (Total_Fuel_for_Boilers)
    _COEF_DMW   = 1.9573            # $/(t/hr DMW)
    _delta_pairs = [
        ("Fuel_Bill",  "Total_Fuel_for_Boilers",    _COEF_FUEL),
        ("Power_Bill", "Total_Power_for_Drives",    _COEF_POWER),
    ]
    for _cost_name, _drv_name, _coef in _delta_pairs:
        _a_cost = actual_ns.get(_cost_name); _a_drv = actual_ns.get(_drv_name); _o_drv = _getopt(_drv_name)
        if _a_cost is None: continue
        if _a_drv is None: _a_drv = actual_ns.get("Total_Fuel_Consumption") if "Fuel" in _cost_name else actual_ns.get("Total_Power_Consumption")
        if _o_drv is None: _o_drv = _getopt("Total_Fuel_Consumption") if "Fuel" in _cost_name else _getopt("Total_Power_Consumption")
        if _a_drv is None or _o_drv is None: continue
        try: _a_cost=float(_a_cost); _a_drv=float(_a_drv); _o_drv=float(_o_drv)
        except Exception: continue
        if _np.isnan(_a_cost) or _np.isnan(_a_drv) or _np.isnan(_o_drv): continue
        _new = _a_cost + (_o_drv - _a_drv) * _coef
        _setopt(_cost_name, _new); _n_fixed += 1
        print(f"    {_cost_name}: {_a_cost:.2f} -> {_new:.2f}  (delta {_drv_name}: {_o_drv-_a_drv:+.4f} x {_coef:.3f})")
    print(f"  [FIX-4c] Cost display recomputed for {_n_fixed} terms")
except Exception as _e:
    print(f"  [FIX-4c] skipped: {_e}")
# [/FIX-4c]

comp_rows = []
ts_now = datetime.now().isoformat()
comp_rows.append({"Variable":"Timestamp","UOMS":"Time","Actual_Data":ts_now,"Optimized_Data":ts_now,"Inferred_Data":ts_now})

# ====================================================================
# FIX: Use GEKKO's true solver objective instead of Python's recalculation
# ====================================================================
try:
    final_opt_obj = solver_obj if (SOLVE_SUCCESS and not np.isnan(solver_obj)) else obj_optimum
except NameError:
    final_opt_obj = obj_optimum

comp_rows.append({
    "Variable": "Objective_Function",
    "UOMS": "$/HR",
    "Actual_Data": obj_actual,
    "Optimized_Data": final_opt_obj,
    "Inferred_Data": actual_ns.get(obj_tag, np.nan)
})
# ====================================================================

for vname in sorted(var_defs.keys()):
    comp_rows.append({"Variable":vname, "UOMS":uom_map.get(vname,""),
        "Actual_Data":actual_ns.get(vname,np.nan),
        "Optimized_Data":opt_vals[vname]["optimum"] if vname in opt_vals else np.nan,
        "Inferred_Data":actual_ns.get(vname,np.nan)})
for tag in sorted(derived_eq_map.keys()):
    if tag not in var_defs:
        comp_rows.append({"Variable":tag, "UOMS":uom_map.get(tag,""),
            "Actual_Data":safe_eval_scalar(derived_eq_map[tag], actual_ns, all_referenced_tags),
            "Optimized_Data":opt_derived.get(tag,np.nan),
            "Inferred_Data":actual_ns.get(tag,np.nan)})
covered = {r["Variable"] for r in comp_rows}
for tag in sorted(inferred_formula_map.keys()):
    if tag in covered: continue
    comp_rows.append({"Variable":tag, "UOMS":uom_map.get(tag,""),
        "Actual_Data":actual_ns.get(tag,np.nan),
        "Optimized_Data":opt_inferred.get(tag,np.nan),
        "Inferred_Data":actual_ns.get(tag,np.nan)})
comparison_df = pd.DataFrame(comp_rows)

# Per-tag completeness summary (Safe fallback for skipped multi-timestamp)
if "_placeholder" in inferred_all_df.columns:
    completeness_df = pd.DataFrame({"Tag": ["N/A"], "Pct_NonNull": [0], "N_NonNull": [0], "N_Total": [0]})
else:
    completeness_df = pd.DataFrame({
        "Tag": completeness.index,
        "Pct_NonNull": completeness.values,
        "N_NonNull": (inferred_all_df.iloc[:,1:].notna().sum()).values,
        "N_Total": len(inferred_all_df),
    }).sort_values("Pct_NonNull", ascending=False).reset_index(drop=True)

ts_file = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")
out_xlsx = f"{OUTPUT_DIR}/{ts_file}_output_v3.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    comparison_df.to_excel(writer, sheet_name="Output_Comparison", index=False)
    inferred_all_df.to_excel(writer, sheet_name="Inferred_All_Timestamps", index=False)
    completeness_df.to_excel(writer, sheet_name="Inferred_Completeness", index=False)
    pre_solve_diag_df.to_excel(writer, sheet_name="Pre_Solve_Diagnostic", index=False)
    HDR = PatternFill("solid", fgColor="1F4E79")
    HFNT = Font(color="FFFFFF", bold=True, name="Arial", size=11)
    BODY = Font(name="Arial", size=10)
    for ws in writer.sheets.values():
        for cell in ws[1]:
            cell.fill = HDR; cell.font = HFNT; cell.alignment = Alignment(horizontal="center")
        for row in ws.iter_rows(min_row=2):
            for cell in row: cell.font = BODY
        for col_cells in ws.columns:
            try:
                mw = max((len(str(c.value)) if c.value else 0) for c in col_cells)
                ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(mw+3, 55)
            except: pass

print(f"  Output file: {out_xlsx}")
print(f"  Sheets: Output_Comparison ({len(comparison_df)} rows)")
print(f"          Inferred_All_Timestamps ({len(inferred_all_df)} rows × {len(inferred_all_df.columns)} cols)")
print(f"          Inferred_Completeness ({len(completeness_df)} rows)")
print(f"          Pre_Solve_Diagnostic ({len(pre_solve_diag_df)} rows)")

print("\n" + "="*70); print("PIPELINE v3 COMPLETE"); print("="*70)
print(f"  Snapshot              : {ts_value}")
try: print(f"  Inferred (snapshot)   : {n_computed} computed, {n_skipped} skipped")
except: pass
print(f"  Inferred (multi-ts)   : {len(inferred_all_df)} timestamps × {len(inferred_formula_map)} formulas")
try: print(f"  Pre-solve violations  : {start_violations} / {n_constraints}")
except: pass
print(f"  GEKKO Solve           : {'SUCCESS' if SOLVE_SUCCESS else 'FAILED'}")
print(f"  Output                : {out_xlsx}")

TASK 9 — EXPORT RESULTS
    Boiler_1_Fuel_Cost: 387.42 -> 331.27  (driver Fuel_BLR_1: 4.5496 -> 3.8902)
    Boiler_2_Fuel_Cost: 387.43 -> 351.26  (driver Fuel_BLR_2: 4.5497 -> 4.1250)
    Boiler_3_Fuel_Cost: 387.42 -> 329.18  (driver Fuel_BLR_3: 4.5496 -> 3.8656)
    Boiler_4_Fuel_Cost: 387.41 -> 351.02  (driver Fuel_BLR_4: 4.5494 -> 4.1221)
    Boiler_5_Fuel_Cost: 397.27 -> 344.05  (driver Fuel_BLR_5: 4.6653 -> 4.0403)
    Costing_total_Fuel_Cost: 1944.88 -> 1793.10  (driver Total_Fuel_Consumption: 36.1462 -> 33.3254)
    DMW_Bill: 460.31 -> 430.61  (driver DMW_Makeup: 235.1720 -> 220.0000)
    Fuel_Bill: 1983.05 -> 1742.88  (delta Total_Fuel_for_Boilers: -2.8204 x 85.155)
    Power_Bill: 2836.81 -> 2846.25  (delta Total_Power_for_Drives: +0.1966 x 48.002)
  [FIX-4c] Cost display recomputed for 9 terms


  Output file: ./output/2026-04-27_15_48_56_output_v3.xlsx
  Sheets: Output_Comparison (3358 rows)
          Inferred_All_Timestamps (1 rows × 1 cols)
          Inferred_Completeness (1 rows)
          Pre_Solve_Diagnostic (38 rows)

PIPELINE v3 COMPLETE
  Snapshot              : 2026-03-31 00:00:00
  Inferred (snapshot)   : 3049 computed, 293 skipped
  Inferred (multi-ts)   : 1 timestamps × 3342 formulas
  Pre-solve violations  : 4 / 38
  GEKKO Solve           : SUCCESS
  Output                : ./output/2026-04-27_15_48_56_output_v3.xlsx


In [11]:
# ====== POST-OPTIMIZER PIPELINE ======
# Runs AFTER the MINLP solver has converged (cells 6-9).
# Inputs:
#   - v6 feature file: authoritative inferred / derived / tag library
#   - feature_file_post_optimizer.xlsx: post-optimizer logic sheets
#   - notebook in-memory state: actual_ns, opt_vals, opt_derived, opt_inferred
# Outputs:
#   - new sheet "Post_Optimizer_Results" appended to the existing output xlsx
#   - separate file post_optimizer_results.xlsx with Per-Tag / SEU / ODS / PEEO sheets
# ----------------------------------------------------------------------
import re as _re, math as _m
import numpy as _np
import pandas as _pd
from openpyxl import load_workbook
from datetime import datetime as _dt

print("\n" + "="*70)
print("TASK 10 - POST-OPTIMIZER PIPELINE")
print("="*70)

POST_FF = "feature_file_eo_v7_unified.xlsx"

# ---------- load post-optimizer sheets ----------
post_der = _pd.read_excel(POST_FF, sheet_name="derived_equation_post_optimizer")
post_map = _pd.read_excel(POST_FF, sheet_name="inferred_tag_rm_block_mapping")
seu_df   = _pd.read_excel(POST_FF, sheet_name="seu_detail")
peeo_df  = _pd.read_excel(POST_FF, sheet_name="peeo_based_adjustment")
sugg_df  = _pd.read_excel(POST_FF, sheet_name="seu_suggestions_mapping")
effect_df= _pd.read_excel(POST_FF, sheet_name="effect")
cause_df = _pd.read_excel(POST_FF, sheet_name="cause")
ods_df   = _pd.read_excel(POST_FF, sheet_name="ods")
out_pi   = _pd.read_excel(POST_FF, sheet_name="output_pi_mapping")
print(f"  Loaded post-optimizer sheets from {POST_FF}")

# ---------- build namespaces ----------
# opt_ns: per-tag optimized value (variable optimum OR derived OR inferred OR actual fallback)
def _val(d, k, default=None):
    v = d.get(k, default)
    if isinstance(v, dict) and "optimum" in v: return v["optimum"]
    return v

actual_map = {k: actual_ns.get(k) for k in actual_ns}
opt_map = {}
for _k in actual_map:
    if   _k in opt_vals:      opt_map[_k] = opt_vals[_k]["optimum"]
    elif _k in opt_derived:   opt_map[_k] = opt_derived[_k]
    elif _k in opt_inferred:  opt_map[_k] = opt_inferred[_k]
    else:                     opt_map[_k] = actual_map[_k]

# add _actual and _optimum suffix views so SEU / ODS expressions resolve
_dual_ns = {}
for _k in set(list(actual_map.keys()) + list(opt_map.keys())):
    _dual_ns[_k + "_actual"]  = actual_map.get(_k)
    _dual_ns[_k + "_optimum"] = opt_map.get(_k)
    _dual_ns[_k] = actual_map.get(_k)      # plain name -> actual (fallback)

# ---------- formula evaluator (same bracket-replacement style as optimizer) ----------
_BRK = _re.compile(r"\[([^\]]+)\]")
def _safe_eval(expr, ns):
    if expr is None or (isinstance(expr, float) and _np.isnan(expr)): return None
    if isinstance(expr, (int, float)): return expr
    s = str(expr).strip()
    if not s: return None
    s = s.replace("&&", " and ").replace("||", " or ")
    # if(cond, a, b)  ->  (a if cond else b)
    def _if_repl(m):
        return f"(({m.group(2)}) if ({m.group(1)}) else ({m.group(3)}))"
    # naive if(...) handling - only top-level
    _if_re = _re.compile(r"if\s*\(([^,()]+(?:\([^)]*\)[^,()]*)*),([^,()]+(?:\([^)]*\)[^,()]*)*),([^()]+(?:\([^)]*\)[^()]*)*)\)")
    for _ in range(4):
        s_new = _if_re.sub(_if_repl, s)
        if s_new == s: break
        s = s_new
    # [tag] -> ns["tag"]
    def _br_repl(m):
        return f"_NS_[\"{m.group(1)}\"]"
    s = _BRK.sub(_br_repl, s)
    try:
        return eval(s, {"__builtins__": {}, "abs": abs, "round": round,
                        "min": min, "max": max, "sum": sum, "len": len,
                        "sqrt": _m.sqrt, "log": _m.log, "exp": _m.exp},
                    {"_NS_": ns})
    except Exception:
        return None

# ---------- STEP 1: recompute post-optimizer inferred tags (propagation) ----------
# Filter inferred-mapping rows that belong to post-optimizer stage.
_post_flag = post_map[post_map["post_optimizer_inferred_calculation"] == 1]["tag_name"].astype(str).tolist()
_post_flag_set = set(_post_flag)

# v6 inferred formula library (user Q5: stay consistent with v6)
_inf_v6 = inferred_formula_map      # already built by earlier cells
n_recomp = 0
for _tag in _post_flag:
    if _tag not in _inf_v6: continue
    _f = _inf_v6[_tag]
    _v = _safe_eval(_f, _dual_ns)
    if _v is None: continue
    # propagate: update opt_map and _dual_ns
    opt_map[_tag] = _v
    _dual_ns[_tag + "_optimum"] = _v
    _dual_ns[_tag] = actual_map.get(_tag, _v)
    n_recomp += 1
print(f"  [Step 1] post-optimizer inferred recomputed (v6 lib): {n_recomp}")

# ---------- STEP 2: derived_equation_post_optimizer (85 rows) ----------
post_der_results = []
n_post_der = 0
# simple passes - 3 rounds to resolve nested dependencies
for _ in range(3):
    for _, _r in post_der.iterrows():
        _tag = str(_r["tag_name"]).strip()
        _exp = _r["formula_expression"]
        _v = _safe_eval(_exp, _dual_ns)
        if _v is None: continue
        opt_map[_tag] = _v
        _dual_ns[_tag + "_optimum"] = _v
        # keep actual fallback if not present
        if _tag + "_actual" not in _dual_ns or _dual_ns[_tag + "_actual"] is None:
            _dual_ns[_tag + "_actual"] = actual_map.get(_tag, _v)
for _, _r in post_der.iterrows():
    _tag = str(_r["tag_name"]).strip()
    _exp = _r["formula_expression"]
    _v_opt = _safe_eval(_exp, _dual_ns)
    _v_act = _safe_eval(_exp, {k: v for k, v in _dual_ns.items() if not k.endswith("_optimum")})
    post_der_results.append({
        "tag_name": _tag, "formula": str(_exp)[:200],
        "actual": actual_map.get(_tag, _v_act),
        "optimum": _v_opt,
    })
    if _v_opt is not None: n_post_der += 1
print(f"  [Step 2] derived_equation_post_optimizer evaluated: {n_post_der}/{len(post_der)}")

# >>>DBG_FURNACE<<<
print('--- FURNACE DEBUG ---')
for _n in range(1,10):
    _s = f'Furnace_{_n}_Status'; _p = f'Furnace_{_n}_Status_PIAF'
    print(f'  {_s}: actual_ns={actual_ns.get(_s)} dual_ns[_actual]={_dual_ns.get(_s+"_actual")} | {_p}: actual_ns={actual_ns.get(_p)}')

# ---------- STEP 3: SEU duties (baseline / actual / target) ----------
seu_results = []
for _, _r in seu_df.iterrows():
    _name = _r["seu_name"]
    _base = _safe_eval(_r.get("baseline_duty_expression"), _dual_ns)
    _act  = _safe_eval(_r.get("actual_duty_expression"),   _dual_ns)
    _tgt  = _safe_eval(_r.get("target_duty_expression"),   _dual_ns)
    _gain = _safe_eval(_r.get("gain_expression"),          _dual_ns)
    _enpi = _safe_eval(_r.get("enpi_expression"),          _dual_ns)
    _bfac = _safe_eval(_r.get("benefit_factor"),           _dual_ns)
    _benefit_usd = None
    try:
        if _enpi is not None and _bfac is not None:
            _benefit_usd = float(_enpi) * float(_bfac)
    except Exception: pass
    seu_results.append({
        "seu_name": _name,
        "display":  _r.get("seu_display_name"),
        "category": _r.get("seu_category"),
        "source":   _r.get("energy_source"),
        "baseline_duty": _base,
        "actual_duty":   _act,
        "target_duty":   _tgt,
        "gain":          _gain,
        "enpi":          _enpi,
        "benefit_factor":_bfac,
        "benefit_$/hr":  _benefit_usd,
    })
_seu_ok = sum(1 for r in seu_results if r["actual_duty"] is not None and r["target_duty"] is not None)
print(f"  [Step 3] SEU duties evaluated: {_seu_ok}/{len(seu_df)}")

# ---------- STEP 4: PEEO step-change suggestions ----------
peeo_results = []
for _, _r in peeo_df.iterrows():
    _parent = _r["parent_tag_name"]; _child = _r["peeo_optimum_tag_name"]
    _step = _r["step_change_value"]; _stype = _r["suggestion_type"]
    _act_val = actual_map.get(_parent)
    _opt_val = opt_map.get(_parent)
    peeo_results.append({
        "parent": _parent, "target_optimum_tag": _child,
        "ods_id": _r.get("peeo_ods_id"),
        "suggestion_type": _stype,
        "step_change": _step,
        "parent_actual": _act_val,
        "parent_optimum": _opt_val,
        "trigger": 1 if (_stype == "yes" and _act_val is not None and _opt_val is not None
                         and not _np.isclose(float(_act_val or 0), float(_opt_val or 0), atol=1e-4)) else 0,
    })
print(f"  [Step 4] PEEO adjustments evaluated: {len(peeo_results)}")

# ---------- STEP 5: ODS engine (cause + effect) ----------
def _fires(expr, ns):
    v = _safe_eval(expr, ns)
    try: return int(float(v) != 0) if v is not None else 0
    except Exception: return 0

effect_fire = {}
effect_rows = []
for _, _r in effect_df.iterrows():
    _name = _r["effect_name"]
    _f = _fires(_r["effect_expression"], _dual_ns)
    effect_fire[_name] = _f
    effect_rows.append({"effect": _name,
        "description": _r.get("effect_description"),
        "fired": _f, "category": _r.get("category"),
        "monitoring_tag": _r.get("monitoring_tag_name", _r.get("monitoring_tag_id", ""))})

cause_fire = {}
cause_rows = []
for _, _r in cause_df.iterrows():
    _name = _r["cause_name"]
    _f = _fires(_r["cause_expression"], _dual_ns)
    cause_fire[_name] = _f
    cause_rows.append({"cause": _name,
        "description": _r.get("cause_description"),
        "fired": _f,
        "message": _r.get("cause_message", _r.get("casue_message", "")),
        "monitoring_tag": _r.get("monitoring_tag_name", _r.get("monitoring_tag_id", ""))})

# ODS alert log: for each effect fired, list causes that also fired
# ods_df is DB-normalized (only cause_id/effect_id); join with effect_df/cause_df for names
_eff_id_to_name = dict(zip(effect_df["effect_id"].astype(int), effect_df["effect_name"].astype(str))) if "effect_id" in effect_df.columns else {}
_cau_id_to_name = dict(zip(cause_df["cause_id"].astype(int),  cause_df["cause_name"].astype(str)))  if "cause_id"  in cause_df.columns else {}
ods_alerts = []
for _, _r in ods_df.iterrows():
    if "effect_name" in _r.index and "cause_name" in _r.index:
        _ef = _r["effect_name"]; _ca = _r["cause_name"]
    else:
        try:    _eid = int(_r["effect_id"]); _cid = int(_r["cause_id"])
        except Exception: continue
        _ef = _eff_id_to_name.get(_eid); _ca = _cau_id_to_name.get(_cid)
        if _ef is None or _ca is None: continue
    if effect_fire.get(_ef, 0) == 1 and cause_fire.get(_ca, 0) == 1:
        _cause_row = cause_df[cause_df["cause_name"] == _ca].iloc[0] if (cause_df["cause_name"] == _ca).any() else None
        _cr = _cause_row.to_dict() if _cause_row is not None else {}
        ods_alerts.append({
            "effect": _ef, "cause": _ca,
            "message": _cr.get("cause_message", _cr.get("casue_message", "")),
            "monitoring_tag": _cr.get("monitoring_tag_name", _cr.get("monitoring_tag_id", "")),
        })
print(f"  [Step 5] ODS effects fired: {sum(effect_fire.values())}/{len(effect_fire)}   "
      f"causes fired: {sum(cause_fire.values())}/{len(cause_fire)}   "
      f"alerts raised: {len(ods_alerts)}")

# ---------- STEP 6: build output frames ----------
post_der_frame = _pd.DataFrame(post_der_results)
seu_frame      = _pd.DataFrame(seu_results)
peeo_frame     = _pd.DataFrame(peeo_results)
effect_frame   = _pd.DataFrame(effect_rows)
cause_frame    = _pd.DataFrame(cause_rows)
ods_frame      = _pd.DataFrame(ods_alerts)

# Consolidated post-optimizer tag frame (everything with actual + optimum)
_all_tags = sorted(set(list(post_der_frame["tag_name"]) + list(_post_flag_set)))
_post_tag_rows = []
for _t in _all_tags:
    _post_tag_rows.append({
        "Variable": _t,
        "Actual_Data":    actual_map.get(_t),
        "Optimized_Data": opt_map.get(_t),
        "Delta":          (opt_map.get(_t) - actual_map.get(_t))
                          if (isinstance(opt_map.get(_t),(int,float))
                              and isinstance(actual_map.get(_t),(int,float))) else None,
        "Category": "post_optimizer_derived" if _t in set(post_der_frame["tag_name"]) else "post_optimizer_inferred",
    })
post_tags_frame = _pd.DataFrame(_post_tag_rows)

# ---------- STEP 7: write outputs ----------
# 7a - append sheet(s) to existing out_xlsx (from cell 9)
try:
    _wb = load_workbook(out_xlsx)
    with _pd.ExcelWriter(out_xlsx, engine="openpyxl", mode="a", if_sheet_exists="replace") as _w:
        post_tags_frame.to_excel(_w, sheet_name="Post_Optimizer_Tags",    index=False)
        post_der_frame.to_excel (_w, sheet_name="Post_Derived_Equations", index=False)
        seu_frame.to_excel      (_w, sheet_name="SEU_Report",             index=False)
        peeo_frame.to_excel     (_w, sheet_name="PEEO_Suggestions",       index=False)
        effect_frame.to_excel   (_w, sheet_name="ODS_Effects",            index=False)
        cause_frame.to_excel    (_w, sheet_name="ODS_Causes",             index=False)
        ods_frame.to_excel      (_w, sheet_name="ODS_Alerts_Log",         index=False)
    print(f"  Appended 7 post-optimizer sheets to {out_xlsx}")
except Exception as _e:
    print(f"  WARN could not append to {out_xlsx}: {_e}")

# 7b - standalone post_optimizer_results.xlsx
_ts = _dt.now().strftime("%Y-%m-%d_%H_%M_%S")
standalone = f"post_optimizer_results_{_ts}.xlsx"
with _pd.ExcelWriter(standalone, engine="openpyxl") as _w:
    post_tags_frame.to_excel(_w, sheet_name="Post_Optimizer_Tags",    index=False)
    post_der_frame.to_excel (_w, sheet_name="Post_Derived_Equations", index=False)
    seu_frame.to_excel      (_w, sheet_name="SEU_Report",             index=False)
    peeo_frame.to_excel     (_w, sheet_name="PEEO_Suggestions",       index=False)
    effect_frame.to_excel   (_w, sheet_name="ODS_Effects",            index=False)
    cause_frame.to_excel    (_w, sheet_name="ODS_Causes",             index=False)
    ods_frame.to_excel      (_w, sheet_name="ODS_Alerts_Log",         index=False)
print(f"  Standalone file: {standalone}")

# ---------- STEP 8: console QC summary ----------
print("\n" + "-"*70)
print("POST-OPTIMIZER QC SUMMARY")
print("-"*70)
print(f"  Post-opt inferred recomputed : {n_recomp}")
print(f"  Post-opt derived evaluated   : {n_post_der}/{len(post_der)}")
print(f"  SEUs with valid duty         : {_seu_ok}/{len(seu_df)}")
print(f"  PEEO step-change rows        : {len(peeo_results)}  "
      f"(triggered: {sum(r['trigger'] for r in peeo_results)})")
print(f"  ODS effects fired            : {sum(effect_fire.values())}/{len(effect_fire)}")
print(f"  ODS causes  fired            : {sum(cause_fire.values())}/{len(cause_fire)}")
print(f"  ODS alert pairs raised       : {len(ods_alerts)}")

# Total SEU benefit (sum over all SEUs where benefit is finite)
_tot = 0.0; _cnt = 0
for r in seu_results:
    b = r.get("benefit_$/hr")
    if isinstance(b, (int, float)) and not _np.isnan(b):
        _tot += float(b); _cnt += 1
print(f"  SEU gross benefit sum        : ${_tot:,.2f}/hr across {_cnt} SEUs")

print("="*70)



TASK 10 - POST-OPTIMIZER PIPELINE


  Loaded post-optimizer sheets from feature_file_eo_v7_unified.xlsx
  [Step 1] post-optimizer inferred recomputed (v6 lib): 87
  [Step 2] derived_equation_post_optimizer evaluated: 72/82
--- FURNACE DEBUG ---
  Furnace_1_Status: actual_ns=0.0 dual_ns[_actual]=0.0 | Furnace_1_Status_PIAF: actual_ns=0.0
  Furnace_2_Status: actual_ns=1.0 dual_ns[_actual]=1.0 | Furnace_2_Status_PIAF: actual_ns=1.0
  Furnace_3_Status: actual_ns=1.0 dual_ns[_actual]=1.0 | Furnace_3_Status_PIAF: actual_ns=1.0
  Furnace_4_Status: actual_ns=1.0 dual_ns[_actual]=1.0 | Furnace_4_Status_PIAF: actual_ns=1.0
  Furnace_5_Status: actual_ns=1.0 dual_ns[_actual]=1.0 | Furnace_5_Status_PIAF: actual_ns=1.0
  Furnace_6_Status: actual_ns=1.0 dual_ns[_actual]=1.0 | Furnace_6_Status_PIAF: actual_ns=1.0
  Furnace_7_Status: actual_ns=1.0 dual_ns[_actual]=1.0 | Furnace_7_Status_PIAF: actual_ns=1.0
  Furnace_8_Status: actual_ns=1.0 dual_ns[_actual]=1.0 | Furnace_8_Status_PIAF: actual_ns=1.0
  Furnace_9_Status: actual_ns=1.0 dual_

  Appended 7 post-optimizer sheets to ./output/2026-04-27_15_48_56_output_v3.xlsx
  Standalone file: post_optimizer_results_2026-04-27_15_48_58.xlsx

----------------------------------------------------------------------
POST-OPTIMIZER QC SUMMARY
----------------------------------------------------------------------
  Post-opt inferred recomputed : 87
  Post-opt derived evaluated   : 72/82
  SEUs with valid duty         : 28/57
  PEEO step-change rows        : 101  (triggered: 0)
  ODS effects fired            : 4/7
  ODS causes  fired            : 1/125
  ODS alert pairs raised       : 1
  SEU gross benefit sum        : $0.00/hr across 0 SEUs
